In [16]:
import os
import sys
import time
import math
from __future__ import annotations

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "cudaq", "matplotlib", "numpy"]
    )
    print("Dependencies installed.")

import cudaq
from cudaq import spin

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Function

import matplotlib
matplotlib.use("Agg")  # no display available on a compute node
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}  |  CUDA-Q {cudaq.__version__}  |  GPUs {cudaq.num_available_gpus()}")

PyTorch 2.14.1  |  CUDA-Q CUDA-Q Version 0.14.2 (https://github.com/NVIDIA/cuda-quantum 91ab3092e76dab8887d1fdf0c99a2478ca90581c)  |  GPUs 0


In [17]:
# --------------------------------------------------------------------
# 1. Experiment parameters
# --------------------------------------------------------------------

# Dataset
POINTS = 96

# Training
EPOCHS = 10
LR = 0.035
WEIGHT_DECAY = 0.0

# Model architecture
HIDDEN = 16
QUBITS = 3

# Quantum simulation
SHOTS = 250
SHIFT = math.pi / 2

# Hardware
DEVICE = "auto"  # "auto", "cpu", "cuda"
CUDAQ_TARGET = "auto"  # "auto", "qpp-cpu", "nvidia", "nvidia-fp64"

# Inference backend
# "same"   -> reuse the training backend
# "qpu"    -> IQM Resonance via IQM_TOKEN and IQM_TARGET_URL
# "hpcqc"  -> LRZ cluster endpoint via IQM_TOKEN and IQM_SERVER_URL (cluster only)
INFERENCE_TARGET = "same"

QPU_MAX_POINTS = 10  # points sent to QPU; each = 1 circuit job

# Reproducibility
SEED_TORCH = 7
SEED_CUDAQ = 11

# Output folder
OUTPUT_DIR = "./qnn-output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"{QUBITS} qubits  |  {SHOTS} shots  |  {EPOCHS} epochs  |  inference = {INFERENCE_TARGET}")

3 qubits  |  250 shots  |  10 epochs  |  inference = same


In [18]:
# --------------------------------------------------------------------
# 2. Configure training runtime
# --------------------------------------------------------------------

torch.manual_seed(SEED_TORCH)
np.random.seed(SEED_TORCH)
cudaq.set_random_seed(SEED_CUDAQ)

if DEVICE == "auto":
    torch_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"DEVICE='auto' selected; using {torch_device}.")
elif DEVICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("DEVICE='cuda' requested but torch.cuda.is_available() is False.")
    torch_device = torch.device("cuda:0")
    print(f"DEVICE='cuda' selected; using {torch_device}.")
else:
    torch_device = torch.device("cpu")
    print(f"DEVICE='cpu' selected; using {torch_device}.")

if CUDAQ_TARGET == "auto":
    if cudaq.num_available_gpus() > 0 and cudaq.has_target("nvidia"):
        cudaq.set_target("nvidia")
        cudaq_target_name = "nvidia"
    else:
        cudaq.set_target("qpp-cpu")
        cudaq_target_name = "qpp-cpu"
else:
    cudaq.set_target(CUDAQ_TARGET)
    cudaq_target_name = CUDAQ_TARGET

print(f"PyTorch device: {torch_device}  |  CUDA-Q target: {cudaq_target_name}")

DEVICE='auto' selected; using cpu.
PyTorch device: cpu  |  CUDA-Q target: qpp-cpu


In [19]:
# --------------------------------------------------------------------
# 3. Dataset
# --------------------------------------------------------------------

x = torch.linspace(-math.pi, math.pi, POINTS, device=torch_device).reshape(-1, 1)
y = torch.sin(x) + 0.3 * torch.cos(3.0 * x)

x_np = x.cpu().numpy().reshape(-1)
y_np = y.cpu().numpy().reshape(-1)

plt.figure(figsize=(8, 3))
plt.plot(x_np, y_np, color="steelblue", lw=2, label="Target")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Training data ({POINTS} points)")
plt.legend()
plt.tight_layout()
plt.savefig("./qnn-output/cudaq-qnn-dataset.png", dpi=160, bbox_inches="tight")

In [20]:
# --------------------------------------------------------------------
# 4. Hybrid model
# --------------------------------------------------------------------

class QuantumCircuitRunner:
    """Owns the CUDA-Q kernel; evaluates shot-based expectation values."""

    def __init__(self, qubits: int, device: torch.device, shift: float, shots: int):
        if qubits < 2:
            raise ValueError("Use at least 2 qubits to include entanglement.")
        self.qubits = qubits
        self.device = device
        self.shift = shift
        self.shots = shots

        # Hamiltonian: sum of Pauli-Z on every qubit (overall magnetisation)
        self.hamiltonian = spin.z(0)
        for i in range(1, qubits):
            self.hamiltonian = self.hamiltonian + spin.z(i)

        @cudaq.kernel
        def kernel(theta: np.ndarray):
            q = cudaq.qvector(qubits)
            # (1) Data encoding
            for i in range(qubits):
                ry(theta[i], q[i])
            # (2) First entangling ring
            for i in range(qubits - 1):
                x.ctrl(q[i], q[i + 1])
            x.ctrl(q[qubits - 1], q[0])
            # (3) Variational layer
            offset = qubits
            for i in range(qubits):
                rz(theta[offset + i], q[i])
            offset += qubits
            for i in range(qubits):
                rx(theta[offset + i], q[i])
            # (4) Second entangling ring
            for i in range(qubits - 1):
                x.ctrl(q[i], q[i + 1])

        self.kernel = kernel

    @property
    def parameter_count(self):
        return 3 * self.qubits

    @property
    def classical_feature_count(self):
        return self.qubits

    @property
    def trainable_quantum_count(self):
        return 2 * self.qubits

    def run(self, theta: torch.Tensor) -> torch.Tensor:
        """Shot-based expectation value (used during training)."""
        theta_np = theta.detach().to("cpu", dtype=torch.float64).numpy()
        results = cudaq.observe(self.kernel, self.hamiltonian, theta_np,
                                 shots_count=self.shots)
        if isinstance(results, list):
            values = [r.expectation() / self.qubits for r in results]
        else:
            values = [results.expectation() / self.qubits]
        return torch.tensor(values, dtype=theta.dtype, device=self.device)

    def run_async(self, theta: torch.Tensor) -> list:
        """Submit one async observe job per sample; returns futures."""
        theta_np = theta.detach().to("cpu", dtype=torch.float64).numpy()
        return [
            cudaq.observe_async(self.kernel, self.hamiltonian, theta_np[i],
                                 shots_count=self.shots)
            for i in range(theta_np.shape[0])
        ]

    def collect_async(self, futures: list, dtype: torch.dtype,
                       device: torch.device) -> torch.Tensor:
        """Block until all futures resolve; return expectation tensor."""
        values = [f.get().expectation() / self.qubits for f in futures]
        return torch.tensor(values, dtype=dtype, device=device)


class QuantumExpectation(Function):
    """Differentiate quantum layer using the parameter-shift rule."""

    @staticmethod
    def forward(ctx, theta: torch.Tensor,
                quantum_circuit: QuantumCircuitRunner) -> torch.Tensor:
        expectation = quantum_circuit.run(theta).reshape(-1, 1)
        ctx.quantum_circuit = quantum_circuit
        ctx.save_for_backward(theta)
        return expectation

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        (theta,) = ctx.saved_tensors
        qc = ctx.quantum_circuit
        grad_theta = torch.zeros_like(theta)
        for i in range(theta.shape[1]):
            t_plus = theta.clone(); t_plus[:, i] += qc.shift
            t_minus = theta.clone(); t_minus[:, i] -= qc.shift
            derivative = 0.5 * (qc.run(t_plus) - qc.run(t_minus))
            grad_theta[:, i] = grad_output.reshape(-1) * derivative
        return grad_theta, None


class QuantumLayer(nn.Module):
    """Wraps QuantumCircuitRunner as a trainable nn.Module."""

    def __init__(self, runner: QuantumCircuitRunner):
        super().__init__()
        self.runner = runner
        self.quantum_weights = nn.Parameter(
            0.10 * torch.randn(runner.trainable_quantum_count, dtype=torch.float32)
        )

    def forward(self, encoded: torch.Tensor) -> torch.Tensor:
        batch = encoded.shape[0]
        weights = self.quantum_weights.to(encoded.device).expand(batch, -1)
        theta = torch.cat([encoded, weights], dim=1)
        return QuantumExpectation.apply(theta, self.runner)


class HybridSineQNN(nn.Module):
    """Hybrid model: classical encoder -> quantum layer -> linear head."""

    def __init__(self, runner: QuantumCircuitRunner, hidden: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(1, hidden),
            nn.Tanh(),
            nn.Linear(hidden, runner.classical_feature_count),
        )
        self.quantum = QuantumLayer(runner)
        self.head = nn.Linear(1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.quantum(self.encoder(x)))

print("Model classes defined.")

Model classes defined.


In [21]:
# --------------------------------------------------------------------
# 5. Instantiate model
# --------------------------------------------------------------------

runner = QuantumCircuitRunner(
    qubits=QUBITS,
    device=torch_device,
    shift=SHIFT,
    shots=SHOTS,
)

model = HybridSineQNN(runner, hidden=HIDDEN).to(torch_device)

classical_params = sum(p.numel() for p in model.encoder.parameters()) + \
    sum(p.numel() for p in model.head.parameters())
quantum_params = model.quantum.quantum_weights.numel()

print(f"Classical params : {classical_params}")
print(f"Quantum params   : {quantum_params}  (Rz + Rx angles)")
print(f"Total params     : {classical_params + quantum_params}")

Classical params : 85
Quantum params   : 6  (Rz + Rx angles)
Total params     : 91


In [22]:
# --------------------------------------------------------------------
# 6. Train
# --------------------------------------------------------------------

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()
losses: list[float] = []

print_every = max(1, EPOCHS // 10)
t0 = time.perf_counter()

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    if torch_device.type == "cuda":
        torch.cuda.synchronize()
    losses.append(float(loss.detach().cpu()))

    if epoch == 0 or (epoch + 1) % print_every == 0:
        elapsed = time.perf_counter() - t0
        print(f"Epoch {epoch+1:4d}/{EPOCHS}  |  MSE = {losses[-1]:.6f}  |  elapsed = {elapsed:.1f}s")

total_time = time.perf_counter() - t0
print(f"Training complete in {total_time:.1f}s ({total_time/EPOCHS:.2f}s/epoch)")

model.eval()
with torch.no_grad():
    final_pred = model(x)
    final_mse = loss_fn(final_pred, y).item()
print(f"Final MSE: {final_mse:.8f}")

Epoch    1/10  |  MSE = 2.035501  |  elapsed = 2.0s
Epoch    2/10  |  MSE = 1.604370  |  elapsed = 4.1s
Epoch    3/10  |  MSE = 1.208841  |  elapsed = 6.2s
Epoch    4/10  |  MSE = 0.921543  |  elapsed = 8.5s
Epoch    5/10  |  MSE = 0.731831  |  elapsed = 10.4s
Epoch    6/10  |  MSE = 0.638700  |  elapsed = 12.4s
Epoch    7/10  |  MSE = 0.581792  |  elapsed = 14.3s
Epoch    8/10  |  MSE = 0.547034  |  elapsed = 16.1s
Epoch    9/10  |  MSE = 0.508396  |  elapsed = 18.1s
Epoch   10/10  |  MSE = 0.474938  |  elapsed = 19.9s
Training complete in 19.9s (1.99s/epoch)
Final MSE: 0.45643568


In [25]:
# --------------------------------------------------------------------
# 7. Results
# --------------------------------------------------------------------

x_np = x.detach().cpu().numpy().reshape(-1)
y_np = y.detach().cpu().numpy().reshape(-1)
pred_np = final_pred.detach().cpu().numpy().reshape(-1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(x_np, y_np, label="Target", color="steelblue", lw=2)
axes[0].plot(x_np, pred_np, label="QNN output", color="tomato", lw=2, linestyle="--")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title(f"Function fit (shots={SHOTS:,})")
axes[0].legend()

axes[1].plot(losses, color="darkorange", lw=1.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].set_title(f"Training loss (final MSE = {final_mse:.5f})")
axes[1].set_yscale("log")

plt.suptitle(
    f"Hybrid QNN  |  {QUBITS} qubits  |  {SHOTS:,} shots  |  {EPOCHS} epochs",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig("./qnn-output/cudaq-qnn-train.png", dpi=160, bbox_inches="tight")

In [24]:
# --------------------------------------------------------------------
# 8. Inference
# --------------------------------------------------------------------

training_target_name = cudaq_target_name

if INFERENCE_TARGET == "same":
    inference_target_name = cudaq_target_name

elif INFERENCE_TARGET == "qpu":
    if not iqm_token or not iqm_target_url:
        raise RuntimeError("Set IQM_TOKEN and IQM_TARGET_URL before using INFERENCE_TARGET='qpu'.")
    os.environ["IQM_TOKEN"] = iqm_token
    cudaq.set_target("iqm", url=iqm_target_url)
    inference_target_name = "IQM Resonance QPU"

elif INFERENCE_TARGET == "hpcqc":
    if not iqm_token or not iqm_server_url:
        raise RuntimeError("Set IQM_TOKEN and IQM_SERVER_URL before using INFERENCE_TARGET='hpcqc'.")
    os.environ["IQM_TOKEN"] = iqm_token
    cudaq.set_target("iqm", url=iqm_server_url)
    inference_target_name = "LRZ HPC-QC"

else:
    raise ValueError(f"INFERENCE_TARGET must be 'same', 'qpu', or 'hpcqc'; got {INFERENCE_TARGET!r}.")

inference_runner = QuantumCircuitRunner(qubits=QUBITS, device=torch_device, shift=SHIFT, shots=SHOTS)
inference_model = HybridSineQNN(inference_runner, hidden=HIDDEN).to(torch_device)
inference_model.load_state_dict(model.state_dict())
inference_model.eval()

print(f"Training backend  : {training_target_name}")
print(f"Inference backend : {inference_target_name}")
print("Weights transferred.")

# Select input points -- subset for QPU to stay within quota, full set for simulators
if INFERENCE_TARGET in ("qpu", "hpcqc"):
    idx_inf = torch.linspace(0, len(x) - 1, QPU_MAX_POINTS).long()
    x_inf = x[idx_inf]
    y_inf = y[idx_inf]
else:
    x_inf, y_inf = x, y

# Classical encoder runs on CPU/GPU
with torch.no_grad():
    encoded = inference_model.encoder(x_inf)
    weights = inference_model.quantum.quantum_weights.expand(encoded.shape[0], -1)
    theta = torch.cat([encoded, weights], dim=1)

# Submit all quantum circuits asynchronously
t_inf = time.perf_counter()
futures = inference_runner.run_async(theta)
print(f"Submitted {len(futures)} job(s) to {inference_target_name} - collecting results...")

# Collect and apply linear head
with torch.no_grad():
    expectation = inference_runner.collect_async(futures, theta.dtype, torch_device)
    inf_pred = inference_model.head(expectation.reshape(-1, 1))
    inf_mse = loss_fn(inf_pred, y_inf).item()
elapsed_inf = time.perf_counter() - t_inf
inf_np = inf_pred.detach().cpu().numpy().reshape(-1)
x_inf_np = x_inf.detach().cpu().numpy().reshape(-1)

print(f"Done in {elapsed_inf:.1f}s  |  MSE = {inf_mse:.6f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_np, y_np, label="Target", color="steelblue", lw=2)
ax.plot(x_np, pred_np, label=f"Trained ({training_target_name})", color="tomato", lw=1.5, linestyle="--")
if INFERENCE_TARGET in ("qpu", "hpcqc"):
    ax.scatter(x_inf_np, inf_np, label=f"Inference ({inference_target_name})",
               color="seagreen", s=60, zorder=5)
else:
    ax.plot(x_inf_np, inf_np, label=f"Inference ({inference_target_name})",
            color="seagreen", lw=1.5, linestyle=":")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(
    f"Backend comparison  |  train MSE = {final_mse:.5f}  inference MSE = {inf_mse:.5f}"
)
ax.legend()
plt.tight_layout()
plt.savefig("./qnn-output/cudaq-qnn-inference.png", dpi=160, bbox_inches="tight")

print("Done.")

Training backend  : qpp-cpu
Inference backend : qpp-cpu
Weights transferred.
Submitted 96 job(s) to qpp-cpu - collecting results...
Done in 1.1s  |  MSE = 0.458116
Done.
